# Hardware Parts Semantic Segmentation

An end-to-end pipeline for multi-class semantic segmentation of hardware parts (hex nuts, washers, bolts, ball bearings, springs, o-rings) using an **Attention Residual U-Net**, trained with a combined Cross-Entropy + Dice loss, evaluated with a competition-style Dice metric, and submitted with **Test-Time Augmentation (TTA)**.

**Pipeline overview:**
1. Imports
2. Configuration & Hyperparameters
3. Dataset & Augmentations
4. Model Architecture (Attention ResUNet)
5. Loss & Metric Functions
6. RLE Encoder & Test-Time Augmentation
7. Train/Validation Split & DataLoaders
8. Model, Loss, Optimizer & Scheduler
9. Training Loop
10. Inference & Submission Generation

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

## 2. Configuration & Hyperparameters

Sets up all paths, hyperparameters, and the class mapping for the 6 hardware-part categories (plus background). Also seeds every RNG for reproducibility and selects the compute device.

In [ ]:
SEED = 42
BATCH_SIZE = 16
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
IMAGE_SIZE = 384
NUM_CLASSES = 7  # 0: Background, 1-6: Hardware Parts
MIN_PIXEL_THRESHOLD = 30  # Threshold to suppress false-positive noise

TRAIN_IMG_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset/train/images"
TRAIN_MASK_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset/train/masks"
TEST_IMG_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset/test/images"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset/sample_submission.csv"
OUTPUT_SUB_PATH = "submission.csv"

CLASS_MAP = {
    1: "hex_nut",
    2: "washer",
    3: "bolt",
    4: "ball_bearing",
    5: "spring",
    6: "o_ring"
}

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Dataset & Augmentations

`HardwareDataset` loads RGB images and their corresponding palette-indexed masks (integer class IDs 0-6). Training augmentations include flips, 90° rotations, affine transforms, and color jitter; validation/test data only get resized and normalized.

In [ ]:
class HardwareDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, image_ids=None, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.image_ids = image_ids if image_ids is not None else sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, img_id)
        image = np.array(Image.open(img_path).convert("RGB"))

        if self.mask_dir is not None:
            mask_path = os.path.join(self.mask_dir, img_id)
            # Palette load yields integer class IDs 0 to 6 directly
            mask = np.array(Image.open(mask_path), dtype=np.int64)

            if self.transform:
                augmented = self.transform(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']
            return image, mask.long(), img_id
        else:
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, img_id

In [ ]:
# Training Augmentations
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.0625, 0.0625), rotate=(-45, 45), p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

## 4. Model Architecture: Attention ResUNet

A U-Net variant that combines:
- **Residual blocks** (`ResidualBlock`) in the encoder/decoder for more stable gradient flow.
- **Attention gates** (`AttentionGate`) on the skip connections, which learn to suppress irrelevant background regions and highlight part-relevant features before concatenation.

The full network (`AttentionResUNet`) is a 4-level encoder-decoder with `base_channels=32`.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x), inplace=True)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

In [ ]:
class AttentionResUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=7, base_channels=32):
        super().__init__()
        c = base_channels
        
        self.inc = ResidualBlock(in_channels, c)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(c, c * 2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(c * 2, c * 4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(c * 4, c * 8))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(c * 8, c * 16))

        self.up1 = nn.ConvTranspose2d(c * 16, c * 8, 2, stride=2)
        self.att1 = AttentionGate(F_g=c * 8, F_l=c * 8, F_int=c * 4)
        self.conv1 = ResidualBlock(c * 16, c * 8)

        self.up2 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.att2 = AttentionGate(F_g=c * 4, F_l=c * 4, F_int=c * 2)
        self.conv2 = ResidualBlock(c * 8, c * 4)

        self.up3 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.att3 = AttentionGate(F_g=c * 2, F_l=c * 2, F_int=c)
        self.conv3 = ResidualBlock(c * 4, c * 2)

        self.up4 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.att4 = AttentionGate(F_g=c, F_l=c, F_int=c // 2)
        self.conv4 = ResidualBlock(c * 2, c)

        self.outc = nn.Conv2d(c, num_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        d1 = self.up1(x5)
        x4_att = self.att1(g=d1, x=x4)
        d1 = torch.cat([d1, x4_att], dim=1)
        d1 = self.conv1(d1)

        d2 = self.up2(d1)
        x3_att = self.att2(g=d2, x=x3)
        d2 = torch.cat([d2, x3_att], dim=1)
        d2 = self.conv2(d2)

        d3 = self.up3(d2)
        x2_att = self.att3(g=d3, x=x2)
        d3 = torch.cat([d3, x2_att], dim=1)
        d3 = self.conv3(d3)

        d4 = self.up4(d3)
        x1_att = self.att4(g=d4, x=x1)
        d4 = torch.cat([d4, x1_att], dim=1)
        d4 = self.conv4(d4)

        return self.outc(d4)

## 5. Loss & Metric Functions

- **`CombinedLoss`**: a weighted sum of standard Cross-Entropy and soft Dice loss, which helps with class imbalance between background and small foreground parts.
- **`calculate_competition_dice`**: computes the mean Dice score across all 6 foreground classes, applying the same minimum-pixel thresholding used at submission time to suppress noisy false positives.

In [ ]:
class CombinedLoss(nn.Module):
    """Combines Cross Entropy with Focal Dice Loss."""
    def __init__(self, num_classes=7, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.num_classes = num_classes
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        smooth = 1e-6
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        dice_loss = 1.0 - torch.mean((2.0 * intersection + smooth) / (cardinality + smooth))

        return (1 - self.dice_weight) * ce_loss + self.dice_weight * dice_loss

In [ ]:
def calculate_competition_dice(pred_masks, gt_masks, min_pixels=MIN_PIXEL_THRESHOLD):
    """Computes competition Dice metric across all images and 6 foreground classes."""
    dice_scores = []
    for pred, gt in zip(pred_masks, gt_masks):
        for cls_id in range(1, 7):
            p_bin = (pred == cls_id)
            g_bin = (gt == cls_id)

            if np.sum(p_bin) < min_pixels:
                p_bin = np.zeros_like(p_bin)

            p_sum = np.sum(p_bin)
            g_sum = np.sum(g_bin)

            if p_sum == 0 and g_sum == 0:
                dice_scores.append(1.0)
            elif p_sum == 0 or g_sum == 0:
                dice_scores.append(0.0)
            else:
                intersection = np.sum(p_bin & g_bin)
                dice_scores.append(2.0 * intersection / (p_sum + g_sum))

    return np.mean(dice_scores)

## 6. RLE Encoder & Test-Time Augmentation

- **`rle_encode`**: Fortran-order (column-major) run-length encoding for submission masks, with a minimum-pixel threshold to skip near-empty predictions.
- **`predict_with_tta`**: averages softmax probabilities across the original image, a horizontal flip, and a vertical flip to produce a more robust prediction.

In [ ]:
def rle_encode(mask, min_pixels=MIN_PIXEL_THRESHOLD):
    """Fortran-order (column-major) RLE encoder."""
    if np.sum(mask) < min_pixels:
        return ""
    
    px = np.asarray(mask, np.uint8).T.flatten()
    px = np.concatenate([[0], px, [0]])
    runs = np.where(px[1:] != px[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(int(x)) for x in runs)

In [ ]:
def predict_with_tta(model, images):
    """Computes averaged predictions across standard & flipped views."""
    # Original
    logits = model(images)
    probs = F.softmax(logits, dim=1)

    # Horizontal Flip
    images_h = torch.flip(images, dims=[3])
    logits_h = model(images_h)
    probs_h = torch.flip(F.softmax(logits_h, dim=1), dims=[3])

    # Vertical Flip
    images_v = torch.flip(images, dims=[2])
    logits_v = model(images_v)
    probs_v = torch.flip(F.softmax(logits_v, dim=1), dims=[2])

    return (probs + probs_h + probs_v) / 3.0

## 7. Train/Validation Split & DataLoaders

Splits the training images 80/20 into train/validation sets and wraps them in `DataLoader`s.

In [ ]:
all_images = sorted(os.listdir(TRAIN_IMG_DIR))

# 80/20 Train/Validation Split
split_idx = int(len(all_images) * 0.8)
train_ids = all_images[:split_idx]
val_ids = all_images[split_idx:]

train_dataset = HardwareDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, train_ids, transform=train_transform)
val_dataset = HardwareDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train images: {len(train_ids)} | Val images: {len(val_ids)}")

## 8. Model, Loss, Optimizer & Scheduler

Instantiates the model, the combined loss, the AdamW optimizer with a cosine-annealing LR schedule, and the AMP gradient scaler for mixed-precision training.

In [ ]:
model = AttentionResUNet(in_channels=3, num_classes=NUM_CLASSES, base_channels=32).to(device)
criterion = CombinedLoss(num_classes=NUM_CLASSES, dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print(model)

## 9. Training Loop

Trains for `NUM_EPOCHS`, validating after every epoch and checkpointing the model whenever validation Dice improves. This is the long-running cell — feel free to lower `NUM_EPOCHS` in the config cell above while iterating.

In [ ]:
best_dice = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0

    for images, masks, _ in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss /= len(train_ids)

    # Validation Phase
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for images, masks, _ in val_loader:
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            val_preds.extend(preds)
            val_targets.extend(masks.numpy())

    val_dice = calculate_competition_dice(val_preds, val_targets)
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Mean Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "best_unet.pth")

print(f"\nBest Validation Mean Dice: {best_dice:.4f}")

## 10. Inference & Submission Generation

Loads the best checkpoint, runs TTA inference on the test set, and writes the RLE-encoded `submission.csv`, aligned exactly to the sample submission's row order.

In [ ]:
print("Generating predictions on Test Set with TTA...")
model.load_state_dict(torch.load("best_unet.pth"))
model.eval()

test_dataset = HardwareDataset(TEST_IMG_DIR, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

predictions = {}

with torch.no_grad():
    for images, img_ids in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            probs = predict_with_tta(model, images)
        preds = torch.argmax(probs, dim=1).cpu().numpy()

        for pred, img_id in zip(preds, img_ids):
            predictions[img_id] = pred

In [ ]:
# Load sample submission to ensure exact key row alignment
sub_df = pd.read_csv(SAMPLE_SUB_PATH)
encoded_rows = []

for _, row in sub_df.iterrows():
    key = row["ImageId_ClassId"]

    cls_id = None
    for c_id, c_name in CLASS_MAP.items():
        if key.endswith("_" + c_name):
            cls_id = c_id
            img_id = key[:-len("_" + c_name)]
            break

    if cls_id is None:
        raise ValueError(f"Could not parse class from submission key: {key}")

    pred_mask = predictions[img_id]
    binary_mask = (pred_mask == cls_id)

    rle_str = rle_encode(binary_mask, min_pixels=MIN_PIXEL_THRESHOLD)
    encoded_rows.append(rle_str)

sub_df["EncodedPixels"] = encoded_rows
sub_df.to_csv(OUTPUT_SUB_PATH, index=False)
print(f"Successfully created '{OUTPUT_SUB_PATH}' with {len(sub_df)} rows.")